# 05 LLM rerank (Colab)

For 300 students with a history and the 100 test new students, the hybrid builds a 20-course shortlist. An LLM then picks 5, puts them in study order and writes a reason for each. I ran it with DeepSeek (`deepseek-flash`, thinking off) and OpenAI (`gpt-4o-mini`).

The prompts are in `llm.py`. Prompt v1 (run2) lost recall against the shortlist's own top 5, so prompt v2 (run3) shows each course's rank and asks the model to stay near the top. I wrote v2 after seeing the v1 test results and ran it once, with no further tuning.

Every returned ID is checked against the shortlist. Answers are saved as they come in, so a run that disconnects can just be started again. Needs `OPENAI_API_KEY` and `DEEPSEEK_API_KEY` in Colab Secrets. Run 03 and 04 first.

In [ ]:
# Step 0: setup (works in Colab and on a laptop)
import os, sys
from pathlib import Path

# Colab: where the project folder sits in Google Drive
DRIVE_PROJECT = "/content/drive/MyDrive/LearnPath"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT)
else:
    PROJECT_DIR = Path.cwd().parent   # running from the notebooks/ folder

assert (PROJECT_DIR / "src" / "learnpath").exists(), f"Project folder not found: {PROJECT_DIR}"
os.environ["LEARNPATH_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Project folder:", PROJECT_DIR)

In [ ]:
# Step 1: install and settings
!pip -q install openai
import pandas as pd
from learnpath import config, experiment as ex, graph, llm, retrieval
from learnpath.hybrid import Recommender

# None = all 400 students. Set to 10 for a quick trial first.
LIMIT = None
RUN_DEEPSEEK = True
RUN_OPENAI = True

In [ ]:
# Step 2: one cheap test call per provider (checks the key and the model name)
# if the model name is wrong, the list shows what the account has
def smoke(provider, model):
    client = llm.make_client(provider)
    try:
        names = sorted(m.id for m in client.models.list().data)
        hint = [n for n in names if any(k in n for k in ("mini", "chat", "flash", "4o"))]
        print(f"{provider}: models that look relevant -> {hint[:15]}")
    except Exception as e:
        print(f"{provider}: could not list models ({e})")
    r = llm.chat(client, model, [{"role": "system", "content": "Reply with JSON."},
                                 {"role": "user", "content": 'Return {"ok": true}'}])
    print(f"{provider} / {model}:", "OK" if r["error"] is None else r["error"], r["text"][:60],
          f"| output tokens: {r['completion_tokens']}")
    return client

clients = {}
if RUN_DEEPSEEK:
    clients["deepseek"] = smoke("deepseek", config.DEEPSEEK_CHAT_MODEL)
if RUN_OPENAI:
    clients["openai"] = smoke("openai", config.OPENAI_CHAT_MODEL)

In [ ]:
# Step 3: shortlists, using the retrieval space that scored best on validation
courses, students, inter, cold, cold_truth = ex.load_data()
PG = graph.build_prereq_graph(courses)
spaces = retrieval.available_spaces(courses)
params = ex.load_params()
missing = [s for s in spaces if s not in params]
assert not missing, f"Run notebook 04 first (not tuned yet: {missing})"
best = ex.best_space(params, spaces)
print("Validation recall@5:", {k: round(v["val_recall@5"], 3) for k, v in params.items()}, "-> using", best)

fit, hist, truth, _ = ex.split_context(inter, "test")
rec = Recommender(courses, spaces[best], PG, fit, alpha=params[best]["alpha"],
                  cold_pop_weight=params[best]["cold_pop_weight"])
course_info = ex.course_stats(courses, fit)

warm_ids = ex.llm_eval_students(students)
warm = students[students["student_id"].isin(warm_ids)]
cold_test, _ = ex.cold_context(cold, cold_truth, "test")
items = ex.build_llm_items(rec, course_info, warm, hist) + ex.build_llm_items(rec, course_info, cold_test, {})
if LIMIT:
    # half from each group, so the trial also covers new students
    n_warm = len(warm)
    items = items[:LIMIT // 2] + items[n_warm:n_warm + LIMIT - LIMIT // 2]
print(len(items), "prompts ready, prompt", config.PROMPT_VERSION, "| saving as", config.LLM_RUN)

In [ ]:
# what the model actually sees for one student
print(items[0]["messages"][1]["content"][:3000])

In [ ]:
# Step 4: run
# uses API credit. DeepSeek first since it's faster
def out_file(provider):
    return config.LLM_DIR / f"rerank_{provider}_{best}_{config.LLM_RUN}.jsonl"

results = {}
if RUN_DEEPSEEK:
    results["deepseek"] = llm.run_batch(items, clients["deepseek"], config.DEEPSEEK_CHAT_MODEL, out_file("deepseek"))
if RUN_OPENAI:
    results["openai"] = llm.run_batch(items, clients["openai"], config.OPENAI_CHAT_MODEL, out_file("openai"))

In [ ]:
# Step 5: quick summary
FULL_RUN = 400   # 300 students with history + 100 new students
for name, df in results.items():
    print(f"\n{name}: {len(df)} answers")
    print("  failed calls:", df["error"].notna().sum())
    print("  invented IDs caught:", int(df["n_invalid"].sum()), "| repeats caught:", int(df["n_duplicate"].sum()),
          "| slots filled from shortlist:", int(df["n_filled"].sum()))
    print("  median seconds per call:", df["latency_s"].median(), "| median output tokens:", df["completion_tokens"].median())
    print(f"  cost so far: ${df['cost_usd'].sum():.3f} | full run would be about ${df['cost_usd'].mean() * FULL_RUN:.2f}")
    ex_ = df.iloc[0]
    print("  example picks:", ex_["recs"])
    for cid, why in list(ex_["reasons"].items())[:2]:
        print("   -", cid, why)

If Colab disconnects, run everything again. Saved answers are reused. Then go to notebook 06.